# V6: Klasifikacija teksta i logistička regresija

**I051 Računalno jezikoslovlje** - vježbe uz _6. predavanje_.

| | Predavanje P6 | Vježbe |
|---|---|---|
| 1 | značajke, sigmoid | tri značajke i klasifikator u nekoliko redaka Pythona |
| 2 | unakrsna entropija, gradijentni spust | učenje težina vlastitim stohastičkim gradijentnim spustom |
| 3 | matrica zabune, preciznost, odziv, F1 | vrednovanje vlastitog modela |
| 4 | vreća riječi, interpretabilnost | `scikit-learn`: model sa 29 572 značajke, što je naučio |
| 5 | (isti postupak, novi zadatak) | analiza sentimenta recenzija filmova (engleski) |

Zadatak je isti kao na predavanju: **je li rečenica iz vijesti portala SETimes ili iz hrvatskih izvora** (domaći portali i web)?

# 0. Priprema

Kao u V4: u VSCode-u otvorite mapu s bilježnicom i odaberite Jupyter kernel. Paketi `scikit-learn` i `nltk` popisani su u `requirements.txt`; ćeliju je dovoljno pokrenuti jednom.

In [ ]:
%pip install -r requirements.txt

U mapi `data` su rečenice korpusa UD Croatian SET (isti korpus kao u prijašnjim vježbama), podijeljene na skup za učenje, razvojni i testni skup. Svaki redak ima dva stupca odvojena tabulatorom:

1. izvor rečenice: `set` (SETimes), `news` (hrvatski portal) ili `web` (hrvatski web),
2. rečenicu.

Funkcija `read_tsv` vraća dvije liste: rečenice i oznake. Portale i web spajamo u jedan razred `hr`.

In [ ]:
import math
import random
import re
from collections import Counter

def read_tsv(path):
    texts, labels = [], []
    for line in open(path, encoding="utf-8"):
        source, text = line.rstrip("\n").split("\t")
        texts.append(text)
        labels.append("set" if source == "set" else "hr")   # dva razreda
    return texts, labels

train_texts, y_train = read_tsv("data/set_train.tsv")
dev_texts, y_dev = read_tsv("data/set_dev.tsv")
test_texts, y_test = read_tsv("data/set_test.tsv")

print(len(train_texts), len(dev_texts), len(test_texts))
print(Counter(y_train), Counter(y_test))
print(train_texts[1], y_train[1])

# 1. Značajke i sigmoid

S predavanja: rečenicu prikazujemo vektorom od tri značajke:

* $x_1$ — broj riječi *kako* (*izjavio je kako …*),
* $x_2$ — ima li rečenica navodnike (1 ili 0),
* $x_3$ — broj riječi prvog lica (*ja, mi, moj, smo …*).

`FIRST` je **skup** (`set`) riječi prvog lica; `w in FIRST` provjerava pripada li riječ skupu.

In [ ]:
FIRST = {"ja", "mi", "me", "mene", "nas", "nam", "moj", "moja",
         "moje", "naš", "naša", "naše", "sam", "smo"}

def features(text):
    words = re.findall(r"\w+", text.lower())
    return [words.count("kako"),                 # x1
            int('"' in text),                     # x2
            sum(w in FIRST for w in words)]       # x3

print(features("NATO je priopćio kako će se pozivnica uputiti čim spor bude riješen."))
print(features("Ali, napokon smo uspjeli pobjeći."))

Logistička regresija računa $z = \mathbf{w} \cdot \mathbf{x} + b$ i vjerojatnost $P(y = 1 \mid x) = \sigma(z)$, gdje je razred 1 = SETimes. Težine su one sa slajda »Težine i pomak«:

In [ ]:
def sigmoid(z):
    return 1 / (1 + math.exp(-z))

def predict(w, b, x):
    z = sum(wi * xi for wi, xi in zip(w, x)) + b    # skalarni umnožak + pomak
    return sigmoid(z)

w, b = [1.25, 2.04, -1.14], -0.62
for text in ["NATO je priopćio kako će se pozivnica uputiti čim spor bude riješen.",
             "Ali, napokon smo uspjeli pobjeći."]:
    print(round(predict(w, b, features(text)), 3), text)

**Pokušajte:** smislite rečenicu za koju model daje vjerojatnost SETimesa veću od 0,9 i rečenicu za koju daje manju od 0,1.

In [ ]:
# vaš kod

# 2. Učenje: unakrsna entropija i gradijentni spust

**Unakrsna entropija** je negativni logaritam vjerojatnosti koju model daje točnom razredu: $-\log \hat y$ ako je $y = 1$, a $-\log(1 - \hat y)$ ako je $y = 0$. Funkcija `loss` računa prosjek na skupu primjera; primjer je par (vektor značajki, oznaka 0 ili 1).

In [ ]:
def loss(w, b, data):
    total = 0
    for x, y in data:
        y_hat = predict(w, b, x)
        total += -math.log(y_hat) if y == 1 else -math.log(1 - y_hat)
    return total / len(data)

data = [(features(t), int(y == "set")) for t, y in zip(train_texts, y_train)]
print(data[:3])
print(loss([0, 0, 0], 0, data))      # sve težine 0: σ(0) = 0,5 za svaku rečenicu

Gubitak $-\log 0{,}5 = 0{,}693$ je polazište. Jedan korak **gradijentnog spusta** mijenja svaku težinu za (pogreška × značajka), pomnoženo stopom učenja `lr` (s predavanja $\eta$):

$$w_j \leftarrow w_j - \eta\,(\hat y - y)\,x_j \qquad b \leftarrow b - \eta\,(\hat y - y)$$

**Stohastički** gradijentni spust radi taj korak za svaki primjer, slučajnim redoslijedom; jedan prolaz kroz sve primjere je **epoha**.

In [ ]:
def sgd_step(w, b, x, y, lr):
    y_hat = predict(w, b, x)
    w = [wi - lr * (y_hat - y) * xi for wi, xi in zip(w, x)]
    b = b - lr * (y_hat - y)
    return w, b

def train(data, epochs=5, lr=0.01):
    w, b = [0] * len(data[0][0]), 0          # onoliko težina koliko ima značajki
    for epoch in range(epochs):
        random.shuffle(data)
        for x, y in data:
            w, b = sgd_step(w, b, x, y, lr)
        print(epoch + 1, [round(v, 2) for v in w], round(b, 2), "gubitak:", round(loss(w, b, data), 3))
    return w, b

random.seed(0)
w, b = train(data)

Težine su blizu onih sa slajda (1,25; 2,04; −1,14; −0,62), a gubitak je pao s 0,693 na 0,608.

**Pokušajte:** pokrenite `train(data, epochs=3, lr=1.0)` i `train(data, epochs=3, lr=0.0001)`. Što se događa s gubitkom kad je stopa učenja prevelika, a što kad je premalena?

In [ ]:
# vaš kod

# 3. Vrednovanje

Rečenicu svrstavamo u SETimes ako je $P(y = 1 \mid x) > 0{,}5$. Funkcija `classify` prima i funkciju značajki, da je poslije možemo zamijeniti.

In [ ]:
def classify(w, b, texts, feats=features):
    return ["set" if predict(w, b, feats(t)) > 0.5 else "hr" for t in texts]

y_pred = classify(w, b, test_texts)
accuracy = sum(p == g for p, g in zip(y_pred, y_test)) / len(y_test)
print("točnost:", round(accuracy, 3))

**Matrica zabune** je samo prebrojavanje parova (zlatna oznaka, odluka sustava). Pozitivan razred je `set`.

In [ ]:
counts = Counter(zip(y_test, y_pred))      # (zlatna, sustav) -> broj
print(counts)

tp = counts[("set", "set")]
fp = counts[("hr", "set")]
fn = counts[("set", "hr")]
precision = tp / (tp + fp)
recall = tp / (tp + fn)
f1 = 2 * precision * recall / (precision + recall)
print(f"P = {precision:.3f}  R = {recall:.3f}  F1 = {f1:.3f}")

Biblioteka `scikit-learn` računa isto za oba razreda, uz makro prosjek:

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred, digits=3))

Usporedite s klasifikatorom koji **uvijek** kaže `set` (s predavanja: »Zamka točnosti«). Parametar `zero_division=0` kaže da preciznost razreda koji se nikad ne predviđa bude 0 umjesto upozorenja.

In [ ]:
print(classification_report(y_test, ["set"] * len(y_test), digits=3, zero_division=0))

### Zadatak 1

Dodajte **četvrtu značajku** po vlastitom izboru (npr. broj glagola izvještavanja *kazao, izjavio, priopćio, rekao, dodao* ili ima li rečenica crticu ` -- `). Naučite model i usporedite makro F1 s modelom s tri značajke (0,600). Je li pomoglo?

In [ ]:
def features4(text):
    words = re.findall(r"\w+", text.lower())
    return features(text) + [0]            # dopunite: zamijenite 0 četvrtom značajkom

data4 = [(features4(t), int(y == "set")) for t, y in zip(train_texts, y_train)]
random.seed(0)
w4, b4 = train(data4)
print(classification_report(y_test, classify(w4, b4, test_texts, features4), digits=3))

# 4. Vreća riječi u scikit-learnu

Umjesto ručnih značajki svaka riječ rječnika postaje jedna značajka (**vreća riječi**). `CountVectorizer` sastavi rječnik iz skupa za učenje (`fit_transform`) i istim rječnikom prebroji riječi u ostalim skupovima (`transform`). Rezultat je rijetka matrica: redak = rečenica, stupac = riječ.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

vectorizer = CountVectorizer()
X_train = vectorizer.fit_transform(train_texts)
X_dev = vectorizer.transform(dev_texts)
X_test = vectorizer.transform(test_texts)
print(X_train.shape)            # (broj rečenica, broj značajki)

Učenje i vrednovanje su po dva retka. `max_iter` je najveći broj koraka optimizacije.

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, digits=3))

Matrica zabune iz biblioteke: retci su zlatne oznake, stupci odluke sustava (redoslijed zadajemo s `labels`).

In [ ]:
from sklearn.metrics import confusion_matrix

confusion_matrix(y_test, y_pred, labels=["set", "hr"])

## 4.1 Što je model naučio?

`model.coef_[0]` su težine riječi. Pozitivne govore u prilog razredu `model.classes_[1]`, negativne razredu `model.classes_[0]`. `argsort()` vraća indekse težina od najmanje prema najvećoj.

In [ ]:
def top_words(model, vectorizer, n=10):
    words = vectorizer.get_feature_names_out()
    order = model.coef_[0].argsort()
    print(model.classes_[0], [words[i] for i in order[:n]])          # najnegativnije
    print(model.classes_[1], [words[i] for i in order[::-1][:n]])    # najpozitivnije

top_words(model, vectorizer)

`predict_proba` daje vjerojatnosti obaju razreda (redoslijedom `model.classes_`). Ovdje se vidi **prečac** s predavanja: domaća vijest dobiva oznaku SETimes.

In [ ]:
for text in ["Vlada je u ponedjeljak raspravljala o cijeni kruha.",
             "Cijena je 15 kuna, a možete je platiti karticom."]:
    print(model.predict_proba(vectorizer.transform([text])).round(3), text)

### Zadatak 2

`LogisticRegression` ima hiperparametar `C`: što je manji, to model jače kažnjava velike težine (regularizacija). Za `C` iz liste `[0.01, 0.1, 1, 10]` naučite model i ispišite makro F1 na **razvojnom** skupu. Za najbolji `C` izračunajte makro F1 na testnom skupu. Funkcija `f1_score(y, y_pred, average="macro")` računa makro F1.

In [ ]:
from sklearn.metrics import f1_score

for C in [0.01, 0.1, 1, 10]:
    m = ...                  # dopunite: model s tim C, naučen na X_train
    print(C, ...)            # dopunite: makro F1 na razvojnom skupu

best = 1                     # dopunite: najbolji C
m = LogisticRegression(C=best, max_iter=1000).fit(X_train, y_train)
print("test:", ...)          # dopunite: makro F1 na testnom skupu

# 5. Isti postupak, novi zadatak: analiza sentimenta

SLP3 kroz cijelo 4. poglavlje koristi **analizu sentimenta**: je li recenzija pozitivna ili negativna? NLTK ima korpus od 2000 recenzija filmova na engleskom (Pang i Lee, 2004.), po 1000 pozitivnih i negativnih. Preuzmimo ga:

In [ ]:
import nltk

nltk.download("movie_reviews")
from nltk.corpus import movie_reviews

ids = movie_reviews.fileids()                 # npr. 'neg/cv000_29416.txt'
print(len(ids), movie_reviews.categories())
print(movie_reviews.raw("pos/cv000_29590.txt")[:300])

Korpus nema podjelu na skupove, pa recenzije slučajno promiješamo i uzmemo 1600 za učenje i 400 za test. Oznaka je kategorija datoteke (`pos` ili `neg`).

In [ ]:
random.seed(0)
random.shuffle(ids)
train_ids, test_ids = ids[:1600], ids[1600:]

mr_train = [movie_reviews.raw(f) for f in train_ids]
mr_y_train = [movie_reviews.categories(f)[0] for f in train_ids]
mr_test = [movie_reviews.raw(f) for f in test_ids]
mr_y_test = [movie_reviews.categories(f)[0] for f in test_ids]
print(Counter(mr_y_train), Counter(mr_y_test))

Ostatak je isti kao u 4. dijelu — samo drugi podaci:

In [ ]:
mr_vectorizer = CountVectorizer()
mr_X_train = mr_vectorizer.fit_transform(mr_train)
mr_X_test = mr_vectorizer.transform(mr_test)

mr_model = LogisticRegression(max_iter=1000)
mr_model.fit(mr_X_train, mr_y_train)
print(classification_report(mr_y_test, mr_model.predict(mr_X_test), digits=3))
top_words(mr_model, mr_vectorizer, n=15)

Riječi su očekivane (*bad, worst, boring* nasuprot *great, excellent, hilarious*), ali ima i iznenađenja.

**Za razmisliti:** među negativnim riječima je *women*, a među pozitivnima *husband*. Što to govori o podacima za učenje? (Predavanje: »Štete i kartica modela«.)

### Zadatak 3

1. Recenzije su duge (prosječno oko 800 tokena), pa ista riječ dolazi mnogo puta. Napravite `CountVectorizer(binary=True)`: značajka je tada samo **pojavljuje li se** riječ (0 ili 1), kao značajka »navodnici« u 1. dijelu. Naučite model ponovno. Kako se promijenila točnost?
2. Novim modelom procijenite rečenice ispod. Koju model pogrešno svrstava i zašto vreća riječi tu ne može pomoći?

In [ ]:
mr_vectorizer = CountVectorizer()      # dopunite: binary=True
mr_X_train = mr_vectorizer.fit_transform(mr_train)
mr_X_test = mr_vectorizer.transform(mr_test)
mr_model = LogisticRegression(max_iter=1000).fit(mr_X_train, mr_y_train)
print(classification_report(mr_y_test, mr_model.predict(mr_X_test), digits=3))

for text in ["A wonderful film with a great cast.",
             "Boring, predictable and far too long.",
             "The movie was not bad at all."]:
    print(...)               # dopunite: vjerojatnosti razreda za text